# Training Model using ANN

In [1]:
# ==========================================
# 1. Import Libraries
# ==========================================

import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ==========================================
# 2. Load Dataset
# ==========================================

df = pd.read_csv("11) Powerplant dataset.csv")

print(df.head())
print(df.shape)


# ==========================================
# 3. Separate Features and Target
# ==========================================

X = df[["AT", "V", "AP", "RH"]]
y = df["PE"]


# ==========================================
# 4. Train Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


# ==========================================
# 5. Feature Scaling
# ==========================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ==========================================
# 6. Convert into Tensors
# ==========================================

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(
    y_train.to_numpy(), dtype=torch.float32
).view(-1, 1)

y_test = torch.tensor(
    y_test.to_numpy(), dtype=torch.float32
).view(-1, 1)


# ==========================================
# 7. TensorDataset
# ==========================================

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)


# ==========================================
# 8. DataLoader
# ==========================================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)


# ==========================================
# 9. Build ANN Model
# Architecture: 4 → 6 → 6 → 1
# ==========================================

model = nn.Sequential(

    nn.Linear(4, 6),
    nn.ReLU(),

    nn.Linear(6, 6),
    nn.ReLU(),

    nn.Linear(6, 1)
)


# ==========================================
# 10. Loss Function
# ==========================================

loss_function = nn.MSELoss()


# ==========================================
# 11. Optimizer
# ==========================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# ==========================================
# 12. Train ANN Model
# ==========================================

epochs = 100

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        # Forward Propagation
        y_pred = model(X_batch)

        # Calculate Loss
        loss = loss_function(y_pred, y_batch)

        # Clear Previous Gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update Weights
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    epoch_loss = total_loss / len(train_dataset)

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1}/{epochs}, "
            f"Loss: {epoch_loss:.4f}"
        )


# ==========================================
# 13. Evaluate Model
# ==========================================

model.eval()

predictions = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        y_pred = model(X_batch)

        predictions.append(y_pred)

y_pred = torch.cat(predictions).squeeze(1).numpy()
y_true = y_test.squeeze(1).numpy()


# ==========================================
# 14. Evaluation Metrics
# ==========================================

mae = mean_absolute_error(y_true, y_pred)

mse = mean_squared_error(y_true, y_pred)

rmse = mse ** 0.5

r2 = r2_score(y_true, y_pred)

print("\nModel Evaluation")
print("-----------------------")
print(f"MAE:  {mae:.4f}")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")


# ==========================================
# 15. Actual vs Predicted
# ==========================================

results = pd.DataFrame({
    "Actual": y_true,
    "Predicted": y_pred
})

print(results.head(10))

      AT      V       AP     RH      PE
0   8.34  40.77  1010.84  90.01  480.48
1  23.64  58.49  1011.40  74.20  445.75
2  29.74  56.90  1007.15  41.91  438.76
3  19.07  49.69  1007.22  76.79  453.09
4  11.80  40.66  1017.13  97.20  464.43
(9568, 5)
Epoch 10/100, Loss: 6555.0678
Epoch 20/100, Loss: 544.5544
Epoch 30/100, Loss: 38.7401
Epoch 40/100, Loss: 22.6291
Epoch 50/100, Loss: 21.6695
Epoch 60/100, Loss: 21.4800
Epoch 70/100, Loss: 21.5380
Epoch 80/100, Loss: 21.4887
Epoch 90/100, Loss: 21.3367
Epoch 100/100, Loss: 21.4095

Model Evaluation
-----------------------
MAE:  3.5466
MSE:  19.6686
RMSE: 4.4349
R2 Score: 0.9313
       Actual   Predicted
0  433.269989  435.567230
1  438.160004  437.553223
2  458.420013  461.634827
3  480.820007  476.609894
4  441.410004  435.912537
5  454.750000  459.333374
6  483.260010  481.417084
7  478.250000  479.352081
8  434.160004  439.830383
9  470.660004  470.354645
